In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import joblib

model_path = "/content/drive/MyDrive/credit_risk_model.pkl"

saved = joblib.load(model_path)

model = saved["model"]
threshold = saved["threshold"]
features = saved["features"]

print("Model loaded successfully!")
print("Threshold:", threshold)
print("Features:", features)

Model loaded successfully!
Threshold: 0.252
Features: ['CreditUtil', 'Age', 'DPD_30_59', 'DebtRatio', 'MonthlyIncome', 'OpenCreditLines', 'DPD_90plus', 'RealEstateLoans', 'DPD_60_89', 'Dependents']


In [7]:
import joblib

drive_path = "/content/drive/MyDrive/"

X_train = joblib.load(drive_path + "X_train.pkl")
X_test = joblib.load(drive_path + "X_test.pkl")

y_train = joblib.load(drive_path + "y_train.pkl")
y_test = joblib.load(drive_path + "y_test.pkl")

print("Datasets loaded successfully!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

Datasets loaded successfully!
X_train: (104999, 10)
X_test: (45000, 10)


In [8]:
import joblib
import shap
import pandas as pd
import numpy as np


# 1. Save background dataset
X_background = X_train[features].sample(
    n=min(50, len(X_train)),
    random_state=42
)

joblib.dump(X_background, "shap_background.pkl")


# 2. Create global SHAP explainer
def predict_default_probability(data):
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data, columns=features)

    return model.predict_proba(data[features])[:, 1]


explainer = shap.Explainer(
    predict_default_probability,
    shap.maskers.Independent(X_background),
    algorithm="permutation",
    feature_names=features
)


# 3. Calculate global SHAP values
X_explain = X_test[features].sample(
    n=min(100, len(X_test)),
    random_state=42
)

global_shap_values = explainer(
    X_explain,
    max_evals=51
)

# Save both the observations and their explanations
joblib.dump(
    {
        "X_explain": X_explain,
        "shap_values": global_shap_values
    },
    "shap_global.pkl"
)

print("SHAP artifacts saved!")

PermutationExplainer explainer: 101it [00:14,  1.64it/s]

SHAP artifacts saved!


In [9]:
from google.colab import files

files.download("shap_background.pkl")
files.download("shap_global.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>